# 투구 제구 성공 확률 모델 V2 학습

Global Long, Global Recent, Seasonless, R/F Expert, Pitcher Expert를 시간순 OOF로 학습하고 비음수 Brier 블렌딩과 walk-forward 보정을 적용합니다. 테스트 행끼리 집계하지 않습니다.

In [1]:
import importlib.util
import importlib.metadata
import subprocess
import sys

required_packages = {
    "catboost": ("catboost", "1.2.10"),
    "sklearn": ("scikit-learn", "1.8.0"),
    "pyarrow": ("pyarrow", "25.0.1"),
}
install = []
for module, (distribution, wanted) in required_packages.items():
    found = importlib.util.find_spec(module) is not None
    current = importlib.metadata.version(distribution) if found else None
    if current != wanted:
        install.append(f"{distribution}=={wanted}")
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install])


In [2]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display
from train_v2 import run_training

ROOT = Path.cwd().resolve()
if not (ROOT / "result.py").exists():
    raise FileNotFoundError("프로젝트 루트에서 main.ipynb를 실행해 주세요.")
RUN_DIR = Path(os.environ.get("BASEBALL_RUN_DIR", ROOT)).resolve()
print(f"project={ROOT}, run_dir={RUN_DIR}")
print(f"task_type={os.environ.get('BASEBALL_TASK_TYPE', 'GPU')}, devices={os.environ.get('BASEBALL_GPU_DEVICES', '0')}")


project=D:\baseball, run_dir=D:\baseball
task_type=GPU, devices=0


## V2 학습 및 OOF 평가

기본값은 전체 데이터와 GPU 0번입니다. 빠른 점검만 할 때는 실행 전에 `BASEBALL_FAST_MODE=1`을 설정하세요. FAST_MODE 결과는 제출에 사용하면 안 됩니다.

In [3]:
result = run_training(ROOT, RUN_DIR)
display(result["metrics"].sort_values(["season", "model"]))
print("Final weights:", result["summary"]["ensemble"]["weight_map"])
print("Calibration:", result["summary"]["calibration"])


0:	learn: 0.6922719	test: 0.6921469	best: 0.6921469 (0)	total: 116ms	remaining: 2m 19s
100:	learn: 0.6783983	test: 0.6799144	best: 0.6799144 (100)	total: 11.2s	remaining: 2m 2s
200:	learn: 0.6762844	test: 0.6799108	best: 0.6799029 (160)	total: 22s	remaining: 1m 49s
bestTest = 0.6799028653
bestIteration = 160
Shrink model to first 161 iterations.
0:	learn: 0.6922980	test: 0.6921163	best: 0.6921163 (0)	total: 141ms	remaining: 2m 48s
100:	learn: 0.6787672	test: 0.6799260	best: 0.6799106 (94)	total: 10.9s	remaining: 1m 59s
200:	learn: 0.6762930	test: 0.6800017	best: 0.6799057 (106)	total: 21.6s	remaining: 1m 47s
bestTest = 0.6799057065
bestIteration = 106
Shrink model to first 107 iterations.
0:	learn: 0.6922981	test: 0.6920680	best: 0.6920680 (0)	total: 103ms	remaining: 2m 3s
100:	learn: 0.6787374	test: 0.6801028	best: 0.6801002 (89)	total: 10.9s	remaining: 1m 58s
200:	learn: 0.6761599	test: 0.6802474	best: 0.6801002 (89)	total: 21.6s	remaining: 1m 47s
bestTest = 0.6801002361
bestIteratio

,season,model,rows,brier,brier_skill_train_prior,log_loss,roc_auc,target_mean,prediction_mean,best_iteration
0,2022,constant_train_prior,247472,0.249366,0.000000,0.691880,0.500000,0.528920,0.543142,0
5,2022,game_type,247472,0.243549,0.023326,0.679920,0.576570,0.528920,0.530050,0
1,2022,global_long,247472,0.243547,0.023333,0.679903,0.576980,0.528920,0.534869,161
2,2022,global_recent,247472,0.243545,0.023344,0.679906,0.577652,0.528920,0.535623,107
3,2022,global_seasonless,247472,0.243623,0.023031,0.680100,0.578307,0.528920,0.531363,90
4,2022,pitcher,247472,0.244109,0.021081,0.681104,0.573348,0.528920,0.529218,87
18,2022,walk_forward_blend,247472,0.243529,0.023408,0.679884,0.577868,0.528920,0.532225,0
19,2022,walk_forward_calibrated,247472,0.243529,0.023408,0.679884,0.577868,0.528920,0.532225,0
6,2023,constant_train_prior,245525,0.251567,0.000000,0.696290,0.500000,0.499957,0.539537,0
11,2023,game_type,245525,0.249140,0.009646,0.691423,0.532945,0.499957,0.504160,0


Final weights: {'global_long': 0.03648300092040782, 'global_recent': 0.0, 'global_seasonless': 0.0, 'game_type': 0.7085826684568582, 'pitcher': 0.2549343306227339}
Calibration: {'version': 2, 'method': 'affine', 'slope': 1.1830065521036663, 'intercept': -0.09988224176613814, 'trained_on_seasons': [2022, 2023, 2024], 'walk_forward_blend_brier': 0.24792432636022568, 'walk_forward_affine_brier': 0.247822904586792}


In [4]:
display(result["correlations"])
display(result["importance"].head(25))
display(pd.read_csv(result["submission_path"]))
print("다음으로 evaluation.ipynb를 실행해 세부 평가를 확인하세요.")


,global_long,global_recent,global_seasonless,game_type,pitcher
global_long,1.000000,0.984807,0.977200,0.856460,0.860466
global_recent,0.984807,1.000000,0.977139,0.856953,0.862376
global_seasonless,0.977200,0.977139,1.000000,0.827610,0.831366
game_type,0.856460,0.856953,0.827610,1.000000,0.830919
pitcher,0.860466,0.862376,0.831366,0.830919,1.000000


,feature,global_long,global_recent,global_seasonless,pitcher_expert,regular_expert,futures_expert,mean_importance
87,season,17.249866,9.849921,0.000000,0.000000,3.204684,29.075325,9.896633
74,recent_game_type_prior,11.960829,8.962346,15.795284,0.000000,1.376332,1.228920,6.553952
55,pitcher_id,4.084780,3.983025,3.985108,4.358268,8.973109,2.801039,4.697555
14,asof_pitcher_prev5_game_success_rate,3.877688,4.784326,4.900785,5.138312,4.282563,2.446560,4.238372
17,asof_pitcher_success_rate,4.450533,5.106157,4.756951,3.296465,6.025341,0.535718,4.028528
84,same_hand,4.338888,5.496936,5.941983,0.000000,5.043005,3.044136,3.977492
3,asof_pitcher_ball_rate,3.311140,3.893539,4.396750,4.812252,4.544592,1.783320,3.790265
75,recent_global_prior,0.814357,1.015230,9.196525,9.509622,0.226132,0.456203,3.536345
72,pitcher_team_id,3.322478,3.369065,3.844496,5.032043,4.105885,0.981516,3.442580
15,asof_pitcher_reverse_rate,2.902437,3.174981,3.676531,3.803427,3.918418,1.479661,3.159243


,row_id,control_success
0,TEST_000001,0.450595
1,TEST_000017,0.431211
2,TEST_000213,0.474463
3,TEST_005332,0.496378
4,TEST_035185,0.475642


다음으로 evaluation.ipynb를 실행해 세부 평가를 확인하세요.
